# mPES - Colab Launcher desde GitHub

Este notebook clona una copia ligera del repositorio en el almacenamiento local
de Colab y ejecuta la optimizacion desde `h1/`. Los resultados se conservan
en Google Drive.

Configura en la primera celda el repositorio y la rama. El clonado usa
`--depth 1 --single-branch` para reducir tiempo, espacio y trafico de red.
Para `ens_sprb` o `ens_accq`, la rama clonada debe incluir los tres modelos en
sus rutas canonicas dentro de `h1/ml/.../inputs/`.

Ejecuta todas las celdas en orden. Para ejecuciones largas, activa Background
execution en la sesion de Colab.

In [2]:
"""Colab launcher for mPES Bayesian optimisation."""
# Mount Google Drive before cloning the repository.
# ==========================================================================
# MOUNT GOOGLE DRIVE
# ==========================================================================
# pyright: reportMissingImports=false
# pylint: disable=import-error,no-name-in-module
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive', force_remount=False)

Mounted at /content/drive


In [4]:
# Configure a Colab optimisation from a shallow Git clone.
import os
import subprocess

REPOSITORY   = 'https://github.com/Maximiliano0/mPES_2026.git'
BRANCH       = 'new_uq'
WORKSPACE    = '/content/mPES'
H_DIR        = os.path.join(WORKSPACE, 'h1')
UTILS_DIR    = os.path.join(WORKSPACE, 'utils')
OUTPUT_ROOT  = '/content/drive/MyDrive/mPES/runs'
PKG          = 'ens_sprb'  # ql | dql | dqn | rdqn | ac | tr | ens_sprb | ens_accq
N_TRIALS     = 50
RESUME_DATE  = ''  # YYYY-MM-DD to resume, or '' for a new run
USE_GPU      = 0

valid_packages = ('ql', 'dql', 'dqn', 'rdqn', 'ac', 'tr', 'ens_sprb', 'ens_accq')
if PKG not in valid_packages:
    raise ValueError(f'Unsupported PKG: {PKG!r}')

os.environ.update({
    'DRIVE_DIR': OUTPUT_ROOT,
    'H_DIR': H_DIR,
    'REPO_DIR': WORKSPACE,
    'WORKSPACE_DIR': WORKSPACE,
    'PKG': PKG,
    'N_TRIALS': str(N_TRIALS),
    'RESUME_DATE': RESUME_DATE,
    'MPES_USE_GPU': str(USE_GPU),
    'MPES_MODEL_ROOT': '',
})
print(f'[INFO] [Celda 1] Configuración guardada: PKG={PKG!r}, N_TRIALS={N_TRIALS}, BRANCH={BRANCH!r}')

[INFO] [Celda 1] Configuración guardada: PKG='ens_sprb', N_TRIALS=50, BRANCH='new_uq'


In [5]:
# Shallow clone: only the selected branch and its current snapshot.
if os.path.isdir(WORKSPACE):
    print(f"[INFO] [Celda 2] Borrando workspace anterior en {WORKSPACE}...")
    subprocess.run(['rm', '-rf', WORKSPACE], check=True)

print(f"[INFO] [Celda 2] Clonando el repositorio rama '{BRANCH}'...")
subprocess.run([
    'git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH,
    REPOSITORY, WORKSPACE,
], check=True)

print("[INFO] [Celda 2] Ejecutando setup_colab.sh (Instalación de dependencias)...")
setup = subprocess.run(
    ['bash', os.path.join(H_DIR, 'general', 'colab', 'setup_colab.sh')],
    check=False,
    env=os.environ.copy(),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(setup.stdout)
if setup.returncode != 0:
    raise RuntimeError(f'setup_colab.sh exited with code {setup.returncode}')
print(f'[INFO] [Celda 2] Shallow clone y setup completados: {REPOSITORY}@{BRANCH}')

[INFO] [Celda 2] Clonando el repositorio rama 'new_uq'...
[INFO] [Celda 2] Ejecutando setup_colab.sh (Instalación de dependencias)...

  mPES  Colab Pro+ bootstrap

 Drive workspace: /content/drive/MyDrive/mPES/runs

  Installing Python dependencies

  Checking pinned optimisation and training dependencies
  Installing pinned runtime packages: gymnasium==1.2.3 matplotlib==3.10.8 numpy==2.4.3 optuna==4.7.0 tensorflow==2.21.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.9/572.9 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━

In [ ]:
# Copy pretrained models from Drive when running an ensemble package.
import shutil

if PKG in ('ens_sprb', 'ens_accq'):
    MODEL_SOURCES = {
        'dqn': os.path.join(OUTPUT_ROOT, 'models', 'dqn_model.keras'),
        'rdqn': os.path.join(OUTPUT_ROOT, 'models', 'rdqn_model.keras'),
        'trf': os.path.join(OUTPUT_ROOT, 'models', 'trf_model.keras'),
    }
    for name, src in MODEL_SOURCES.items():
        if not os.path.isfile(src):
            raise FileNotFoundError(
                f'Modelo {name} no encontrado en {src}. '
                'Sube los .keras a Drive en esa ruta o ajusta MODEL_SOURCES.'
            )
    for name, src in MODEL_SOURCES.items():
        dst = os.path.join(H_DIR, 'ml', f'pes_{name}', 'inputs', f'{name}_model.keras')
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
        print(f'[INFO] [Celda 3] Modelo copiado: {name} → {dst}')
else:
    print('[INFO] [Celda 3] PKG no es un ensemble; se omite la copia de modelos.')

In [ ]:
run_environment = os.environ.copy()
run_environment.update({
    'DRIVE_DIR': OUTPUT_ROOT,
    'H_DIR': H_DIR,
    'REPO_DIR': WORKSPACE,
    'WORKSPACE_DIR': WORKSPACE,
    'WORKSPACE': WORKSPACE,
    'PKG': PKG,
    'N_TRIALS': str(N_TRIALS),
    'RESUME_DATE': RESUME_DATE,
    'MPES_USE_GPU': str(USE_GPU),
    'MPES_MODEL_ROOT': '',
})
_script = '''
set -uo pipefail
cd "$WORKSPACE"
source /content/mpes_env.sh
bash "$H_DIR/general/colab/run_colab.sh" "$PKG" "$N_TRIALS" "$RESUME_DATE"
'''
print("[INFO] [Celda 4] Iniciando run_colab.sh... Observa los registros a continuación:")
run = subprocess.run(
    _script,
    shell=True,
    executable='/bin/bash',
    check=False,
    env=run_environment,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(run.stdout)
if run.returncode != 0:
    print(f"[ERROR] [Celda 4] run_colab.sh falló con código {run.returncode}")
    print(f"[ERROR] Revisa: {OUTPUT_ROOT}/pes_ens_sprb/<FECHA>_BAYESIAN_OPT/")
    raise RuntimeError(f'run_colab.sh exited with code {run.returncode}')
print("[INFO] [Celda 4] Ejecución de run_colab.sh completada con éxito!")

[INFO] [Celda 3] Iniciando run_colab.sh... Observa los registros a continuación:

  Launching Bayesian optimisation on Colab Pro+

  Package         : pes_ens_sprb
  Module          : ens.pes_ens_sprb.ext.optimize_ens
  Trials          : 50
  Run date        : 2026-09-03
  Output dir      : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT
  Storage         : sqlite:////content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/optuna_study_2026-09-03.db
  Git             : new_uq@b6387c4
  Python          : 3.13.15
  GPU mode        : 0

 Optimisation PID    : 2523
 stdout              : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/bayesian_opt.log
 stderr              : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/bayesian_opt_err.log
 metadata            : /content/drive/MyDrive/mPES/runs/pes_ens_sprb/2026-09-03_BAYESIAN_OPT/run_meta.json


  Optimisation supervisor launched.
   Cell BLOCKS in foreground